# Emerging Tech Lab - Solvent-Screen Opentrons Protocol

## Solvent-screen protocol setup

In [3]:
from solvent_screen_opentrons_helpers import (
    setup_run_logger,
    get_run_log_path,
    DEFAULT_TOUCH_TIP_SPEED,
    DEFAULT_TOUCH_TIP_RADIUS,
    DEFAULT_TOUCH_TIP_V_OFFSET,
    log_step,
    validate_touch_tip_clearance,
    load_source_plates,
    build_solvent_screen_plate_map,
    validate_volume_for_p300,
    dispense_to_wells_with_tip_changes,
    set_robot_speeds,
    log_absolute_time_tick,
    summarise_solvent_screen_records,
)

run_log_path = setup_run_logger(
    log_prefix="solvent_screen_opentrons_run_log",
    log_dir="opentrons_logs",
)
log_step(None, f"Run log initialised: {run_log_path}")

import opentrons.execute
protocol = opentrons.execute.get_protocol_api("2.19")

# Loading labware
plate_48 = protocol.load_labware(
    "greenaway_48_wellplate_3750ul",
    location=2,
)
pip_rack = protocol.load_labware(
    "opentrons_96_tiprack_300ul",
    location=6,
)

# Load one or more 8-well source plates.
# Deck slots must not overlap with the 48-well plate, tip rack, or other labware.
# To add a third source plate, use SOURCE_PLATE_LOCATIONS = [3, 4, 5]
# and then define plate_8_3 = source_plates["plate_8_3"].
SOURCE_PLATE_LOCATIONS = [1, 3, 5]
source_plates = load_source_plates(
    protocol=protocol,
    labware_name="greenaway_8_wellplate_20000ul",
    plate_locations=SOURCE_PLATE_LOCATIONS,
    name_prefix="plate_8",
)
plate_8_1 = source_plates["plate_8_1"]
plate_8_2 = source_plates["plate_8_2"]
plate_8_3 = source_plates["plate_8_3"]

# Load pipette
pip_300 = protocol.load_instrument(
    "p300_single_gen2",
    "left",  # change to "right" if needed
    tip_racks=[pip_rack],
)

# -----------------------------
# User setup: liquid handling
# -----------------------------

ASPIRATE_RATE = 70
STANDARD_DISPENSE_RATE = 70
SLOW_DISPENSE_RATE = 10  # used for dropwise dialdehyde addition
AIR_GAP_VOLUME = 15
MAX_DISPENSE = 200
PRE_WET_CYCLES = 3
PRE_WET_VOLUME = 180
TIP_CHANGE_INTERVAL = 3
TRANSFER_MODE = "fast"  # "fast" = one tip per reagent/solvent source; "accurate" = change tips regularly
TOUCH_TIP_SPEED = DEFAULT_TOUCH_TIP_SPEED  # mm/s; keep sanity-check and pre-wet rim-touch speed aligned
TOUCH_TIP_RADIUS = DEFAULT_TOUCH_TIP_RADIUS  # keep below 1.0 to avoid full-edge moves in large source wells
TOUCH_TIP_V_OFFSET = DEFAULT_TOUCH_TIP_V_OFFSET  # mm below well top

pip_300.flow_rate.aspirate = ASPIRATE_RATE
pip_300.flow_rate.dispense = STANDARD_DISPENSE_RATE

# Speed up robot movement while keeping liquid-handling flow rates controlled.
# Rim-touching during pre-wetting is slowed separately inside the helper function.
set_robot_speeds(
    protocol=protocol,
    pipette=pip_300,
    pipette_default_speed=400,
)

# -----------------------------
# User setup: source locations and target layout
# -----------------------------

# Source plates hold reagent stocks. The 48-well plate holds reaction conditions.
# 8-well source plate example layout:
# plate_8_1 A1-A2: diamine stocks in different solvents
# plate_8_1 B1-B2: dialdehyde stocks in matching solvents
# plate_8_2 A1/A4: additional diamine stocks
# plate_8_2 B1/B4: additional dialdehyde stocks
# To add more solvent conditions, add entries below and point their source wells
# to whichever loaded source plate contains that stock.
#
# Solvent-screen target layout:
# Each solvent condition is run in triplicate.
# Replicates are arranged vertically within one column block.
#
# Conditions 1-8 use the upper half of the 48-well plate:
#   condition 1 -> A1, B1, C1
#   condition 2 -> A2, B2, C2
#   ...
#   condition 8 -> A8, B8, C8
#
# Conditions 9-16 use the lower half:
#   condition 9  -> D1, E1, F1
#   condition 10 -> D2, E2, F2
#   ...
#   condition 16 -> D8, E8, F8
SOLVENT_CONDITIONS = {
    "CHCl3": {
        "diamine_source": plate_8_1["A1"],
        "dialdehyde_source": plate_8_1["B1"],
    },
    "MeOH": {
        "diamine_source": plate_8_1["A2"],
        "dialdehyde_source": plate_8_1["B2"],
    },
    "THF": {
        "diamine_source": plate_8_1["A3"],
        "dialdehyde_source": plate_8_1["B3"],
    },
    "MeTHF": {
        "diamine_source": plate_8_1["A4"],
        "dialdehyde_source": plate_8_1["B4"],
        # Optional override example:
        # "target_index": 9,  # would force D1, E1, F1 regardless of list order
    },
    "DCM": {
        "diamine_source": plate_8_2["A1"],
        "dialdehyde_source": plate_8_2["B1"],
    },
    "CDCl3": {
        "diamine_source": plate_8_2["A2"],
        "dialdehyde_source": plate_8_2["B2"],
    },
    "EtOH": {
        "diamine_source": plate_8_2["A3"],
        "dialdehyde_source": plate_8_2["B3"],
    },
    "toluene": {
        "diamine_source": plate_8_2["A4"],
        "dialdehyde_source": plate_8_2["B4"],
    },
    "hexane": {
        "diamine_source": plate_8_3["A1"],
        "dialdehyde_source": plate_8_3["B1"],
    },
    "MeCN": {
        "diamine_source": plate_8_3["A2"],
        "dialdehyde_source": plate_8_3["B2"],
    },
}

# Select the solvent condition(s) for this run.
# The order of this list controls target-well assignment unless a condition defines target_index.
CONDITIONS_TO_RUN = [
    "CHCl3",
    "MeOH",
    "THF",
    "MeTHF",
    "DCM",
    "CDCl3",
    "EtOH",
    "toluene",
    "hexane",
    "MeCN"
]

if len(CONDITIONS_TO_RUN) == 0:
    raise ValueError("Select at least one solvent condition to run.")

# -----------------------------
# User setup: dispense volumes
# -----------------------------

# Enter the calculated volume for each stock solution per reaction vial.
# These volumes are applied to every replicate well in this run.
volume_of_diamine = 200  # uL per well
volume_of_dialdehyde = 40  # uL per well

# -----------------------------
# Build and validate the plate map
# -----------------------------

validate_volume_for_p300(volume_of_diamine, "Diamine", protocol=protocol)
validate_volume_for_p300(volume_of_dialdehyde, "Dialdehyde", protocol=protocol)

condition_plate_map = build_solvent_screen_plate_map(
    solvent_conditions=SOLVENT_CONDITIONS,
    conditions_to_run=CONDITIONS_TO_RUN,
    protocol=protocol,
)

Run log file: /var/lib/jupyter/notebooks/Untitled Folder/opentrons_logs/solvent_screen_opentrons_run_log_20260518_084052.txt
Run log initialised: /var/lib/jupyter/notebooks/Untitled Folder/opentrons_logs/solvent_screen_opentrons_run_log_20260518_084052.txt
Loaded source plate plate_8_1 in deck slot 1.
Loaded source plate plate_8_2 in deck slot 3.
Loaded source plate plate_8_3 in deck slot 5.
Set pipette default movement speed to 400 mm/s.
Condition: CHCl3 -> target index 1: A1, B1, C1
Condition: MeOH -> target index 2: A2, B2, C2
Condition: THF -> target index 3: A3, B3, C3
Condition: MeTHF -> target index 4: A4, B4, C4
Condition: DCM -> target index 5: A5, B5, C5
Condition: CDCl3 -> target index 6: A6, B6, C6
Condition: EtOH -> target index 7: A7, B7, C7
Condition: toluene -> target index 8: A8, B8, C8
Condition: hexane -> target index 9: D1, E1, F1
Condition: MeCN -> target index 10: D2, E2, F2


## Basic OT-2 sanity test

In [5]:
# -----------------------------
# Basic OT-2 sanity test
# -----------------------------

protocol.home()
if getattr(pip_300, "has_tip", False):
    pip_300.drop_tip()
    log_step(protocol, "Dropped existing attached tip before sanity test.")

source_wells_to_check = []
for source_plate_name, source_plate in source_plates.items():
    for source_well in source_plate.wells():
        source_wells_to_check.append((source_plate_name, source_well))

validate_touch_tip_clearance(
    wells=[source_well for _, source_well in source_wells_to_check],
    touch_tip_radius=TOUCH_TIP_RADIUS,
)
log_step(protocol, "Source touch-tip XY clearance check passed.")

log_step(protocol, "Starting basic OT-2 sanity test.")

# Test tip pickup/drop
pip_300.pick_up_tip()
log_step(protocol, "Picked up one tip successfully.")

def _well_name(well):
    return getattr(well, "well_name", str(well))


# Test movement to the top of every loaded source well.
for source_plate_name, source_well in source_wells_to_check:
    source_well_name = _well_name(source_well)
    pip_300.move_to(source_well.top())
    log_step(protocol, f"Moved to source {source_plate_name} {source_well_name} top.")

# Test movement to the top of every 48-well reaction-plate well.
for target_well in plate_48.wells():
    target_well_name = _well_name(target_well)
    pip_300.move_to(target_well.top())
    log_step(protocol, f"Moved to target well {target_well_name} top.")

# Test the same slow source rim-touch path used by pre-wetting.
# This catches edge wells that are reachable at the center but unsafe during touch_tip().
for source_plate_name, source_well in source_wells_to_check:
    source_well_name = _well_name(source_well)
    original_default_speed = getattr(pip_300, "default_speed", None)
    try:
        if original_default_speed is not None:
            pip_300.default_speed = TOUCH_TIP_SPEED
        pip_300.touch_tip(
            source_well,
            radius=TOUCH_TIP_RADIUS,
            v_offset=TOUCH_TIP_V_OFFSET,
            speed=TOUCH_TIP_SPEED,
        )
    finally:
        if original_default_speed is not None:
            pip_300.default_speed = original_default_speed
    log_step(protocol, f"Touched source {source_plate_name} {source_well_name} rim successfully.")

pip_300.drop_tip()
log_step(protocol, "Dropped tip successfully.")

protocol.home()
log_step(protocol, "Basic OT-2 sanity test complete.")

Dropped existing attached tip before sanity test.
Source touch-tip XY clearance check passed.
Starting basic OT-2 sanity test.
Picked up one tip successfully.
Moved to source plate_8_1 A1 top.
Moved to source plate_8_1 B1 top.
Moved to source plate_8_1 A2 top.
Moved to source plate_8_1 B2 top.
Moved to source plate_8_1 A3 top.
Moved to source plate_8_1 B3 top.
Moved to source plate_8_1 A4 top.
Moved to source plate_8_1 B4 top.
Moved to source plate_8_2 A1 top.
Moved to source plate_8_2 B1 top.
Moved to source plate_8_2 A2 top.
Moved to source plate_8_2 B2 top.
Moved to source plate_8_2 A3 top.
Moved to source plate_8_2 B3 top.
Moved to source plate_8_2 A4 top.
Moved to source plate_8_2 B4 top.
Moved to source plate_8_3 A1 top.
Moved to source plate_8_3 B1 top.
Moved to source plate_8_3 A2 top.
Moved to source plate_8_3 B2 top.
Moved to source plate_8_3 A3 top.
Moved to source plate_8_3 B3 top.
Moved to source plate_8_3 A4 top.
Moved to source plate_8_3 B4 top.
Moved to target well A1 t

Out of bounds move: X=(418.8192772500001 motor controller, 418.23375000000004 deck) too high for limit 418.0


Touched source plate_8_2 A4 rim successfully.


Out of bounds move: X=(418.8192772500001 motor controller, 418.23375000000004 deck) too high for limit 418.0


Touched source plate_8_2 B4 rim successfully.
Touched source plate_8_3 A1 rim successfully.
Touched source plate_8_3 B1 rim successfully.
Touched source plate_8_3 A2 rim successfully.
Touched source plate_8_3 B2 rim successfully.
Touched source plate_8_3 A3 rim successfully.
Touched source plate_8_3 B3 rim successfully.
Touched source plate_8_3 A4 rim successfully.
Touched source plate_8_3 B4 rim successfully.
Dropped tip successfully.
Basic OT-2 sanity test complete.


## Automated solvent-screen execution

In [6]:
# -----------------------------
# Automated solvent-screen execution
# Correct addition order:
#   1. diamine solution
#   2. dialdehyde solution, slow/dropwise
# -----------------------------

# Store dispense records for later inspection.
diamine_dispense_records = {}
dialdehyde_dispense_records = {}

# Add diamine solution to all replicate wells for each solvent condition.
# This does not start the imine reaction until dialdehyde is added.
for condition_name in CONDITIONS_TO_RUN:
    condition = condition_plate_map[condition_name]
    diamine_dispense_records[condition_name] = dispense_to_wells_with_tip_changes(
        pipette=pip_300,
        protocol=protocol,
        plate=plate_48,
        source_well=condition["diamine_source"],
        target_well_names=condition["target_wells"],
        total_volume=volume_of_diamine,
        dispense_rate=STANDARD_DISPENSE_RATE,
        reagent_name=f"diamine stock ({condition_name})",
        tip_change_interval=TIP_CHANGE_INTERVAL,
        transfer_mode=TRANSFER_MODE,
        max_dispense=MAX_DISPENSE,
        air_gap_volume=AIR_GAP_VOLUME,
        pre_wet_cycles=PRE_WET_CYCLES,
        pre_wet_volume=PRE_WET_VOLUME,
        touch_tip_speed=TOUCH_TIP_SPEED,
        touch_tip_radius=TOUCH_TIP_RADIUS,
        touch_tip_v_offset=TOUCH_TIP_V_OFFSET,
        log_each_dispense_time=False,
        log_absolute_time=False,
    )

# Add dialdehyde solution slowly/dropwise to start the reaction for each solvent condition.
for condition_name in CONDITIONS_TO_RUN:
    condition = condition_plate_map[condition_name]
    log_absolute_time_tick(protocol, f"Starting dialdehyde addition for solvent-screen condition {condition_name}.")

    dialdehyde_dispense_records[condition_name] = dispense_to_wells_with_tip_changes(
        pipette=pip_300,
        protocol=protocol,
        plate=plate_48,
        source_well=condition["dialdehyde_source"],
        target_well_names=condition["target_wells"],
        total_volume=volume_of_dialdehyde,
        dispense_rate=SLOW_DISPENSE_RATE,
        reagent_name=f"dialdehyde stock ({condition_name})",
        tip_change_interval=TIP_CHANGE_INTERVAL,
        transfer_mode=TRANSFER_MODE,
        max_dispense=MAX_DISPENSE,
        air_gap_volume=AIR_GAP_VOLUME,
        pre_wet_cycles=PRE_WET_CYCLES,
        pre_wet_volume=PRE_WET_VOLUME,
        touch_tip_speed=TOUCH_TIP_SPEED,
        touch_tip_radius=TOUCH_TIP_RADIUS,
        touch_tip_v_offset=TOUCH_TIP_V_OFFSET,
        log_each_dispense_time=True,
        log_absolute_time=True,
    )

    log_absolute_time_tick(protocol, f"Finished dialdehyde addition for solvent-screen condition {condition_name}.")

# Reset dispense rate and summarise the run.
pip_300.flow_rate.dispense = STANDARD_DISPENSE_RATE

log_step(protocol, "Solvent-screen summary by condition:")
summarise_solvent_screen_records(
    dispense_records_by_condition=dialdehyde_dispense_records,
    protocol=protocol,
)

protocol.home()

Fast mode enabled for diamine stock (CHCl3): using one tip for this reagent/solvent source.
Picked up tip 1 for diamine stock (CHCl3).
Adding 200 uL diamine stock (CHCl3) to A1.
Adding 200 uL diamine stock (CHCl3) to B1.
Adding 200 uL diamine stock (CHCl3) to C1.
Dropped tip 1 after 3 target-well dispense(s) for diamine stock (CHCl3).
Fast mode enabled for diamine stock (MeOH): using one tip for this reagent/solvent source.
Picked up tip 1 for diamine stock (MeOH).
Adding 200 uL diamine stock (MeOH) to A2.
Adding 200 uL diamine stock (MeOH) to B2.
Adding 200 uL diamine stock (MeOH) to C2.
Dropped tip 1 after 3 target-well dispense(s) for diamine stock (MeOH).
Fast mode enabled for diamine stock (THF): using one tip for this reagent/solvent source.
Picked up tip 1 for diamine stock (THF).
Adding 200 uL diamine stock (THF) to A3.
Adding 200 uL diamine stock (THF) to B3.
Adding 200 uL diamine stock (THF) to C3.
Dropped tip 1 after 3 target-well dispense(s) for diamine stock (THF).
Fast mo

Out of bounds move: X=(418.8192772500001 motor controller, 418.23375000000004 deck) too high for limit 418.0


Adding 200 uL diamine stock (toluene) to A8.
Adding 200 uL diamine stock (toluene) to B8.
Adding 200 uL diamine stock (toluene) to C8.
Dropped tip 1 after 3 target-well dispense(s) for diamine stock (toluene).
Fast mode enabled for diamine stock (hexane): using one tip for this reagent/solvent source.
Picked up tip 1 for diamine stock (hexane).
Adding 200 uL diamine stock (hexane) to D1.
Adding 200 uL diamine stock (hexane) to E1.
Adding 200 uL diamine stock (hexane) to F1.
Dropped tip 1 after 3 target-well dispense(s) for diamine stock (hexane).
Fast mode enabled for diamine stock (MeCN): using one tip for this reagent/solvent source.
Picked up tip 1 for diamine stock (MeCN).
Adding 200 uL diamine stock (MeCN) to D2.
Adding 200 uL diamine stock (MeCN) to E2.
Adding 200 uL diamine stock (MeCN) to F2.
Dropped tip 1 after 3 target-well dispense(s) for diamine stock (MeCN).
[2026-05-18 09:02:51] Starting dialdehyde addition for solvent-screen condition CHCl3.
Fast mode enabled for dialdeh

Out of bounds move: X=(418.8192772500001 motor controller, 418.23375000000004 deck) too high for limit 418.0


[2026-05-18 09:20:36] Starting dispense: 40 uL dialdehyde stock (toluene) to A8.
[2026-05-18 09:20:48] Finished dispense: 40 uL dialdehyde stock (toluene) to A8.
[2026-05-18 09:20:48] Starting dispense: 40 uL dialdehyde stock (toluene) to B8.
[2026-05-18 09:21:01] Finished dispense: 40 uL dialdehyde stock (toluene) to B8.
[2026-05-18 09:21:01] Starting dispense: 40 uL dialdehyde stock (toluene) to C8.
[2026-05-18 09:21:15] Finished dispense: 40 uL dialdehyde stock (toluene) to C8.
Dropped tip 1 after 3 target-well dispense(s) for dialdehyde stock (toluene).
[2026-05-18 09:21:28] Finished dialdehyde addition for solvent-screen condition toluene.
[2026-05-18 09:21:28] Starting dialdehyde addition for solvent-screen condition hexane.
Fast mode enabled for dialdehyde stock (hexane): using one tip for this reagent/solvent source.
Picked up tip 1 for dialdehyde stock (hexane).
[2026-05-18 09:23:01] Starting dispense: 40 uL dialdehyde stock (hexane) to D1.
[2026-05-18 09:23:13] Finished dispe